# 09 自己蒸留（疑似ラベルの上限を画像側から超える）
1. 画像モデルを NFOLD 分割で学習し、各スタディを「そのスタディを学習に使っていないモデル」で予測する（OOF）。
2. LLMラベルと OOF 予測を ALPHA で混ぜて、ラベルを作り直す。
3. 新ラベルで本学習をやり直す（`ft_model_f_s*.pt`）。58件の正解ラベルは評価専用。
04 と同じキャッシュ・設定（設定C）。58件のAUCは、分割モデルの平均（元ラベル）と、本学習（新ラベル）を並べて出す。

In [ ]:
import os, sys, subprocess, warnings
os.environ['PYTHONWARNINGS'] = 'ignore'  # joblibのワーカープロセスにも効かせる
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
from pathlib import Path
from joblib import Parallel, delayed
_name = 'rsna-knee-abnormality-detection'
_cands = [Path('/kaggle/input/competitions')/_name, Path('/kaggle/input')/_name]
COMP = next((d for d in _cands if (d/'train.csv').exists()), None)
assert COMP, f"train.csv が見つからない。/kaggle/input: {[p.name for p in Path('/kaggle/input').iterdir()]}"
print('COMP =', COMP)
REPO = Path('/kaggle/temp/repo') if Path('/kaggle/temp').exists() else Path('/tmp/repo')   # /kaggle/working に置くと実行の出力に含まれてしまう
SRC = REPO/'rsna-knee-abnormality-detection'/'src'
if not SRC.exists():  # internet ON のときだけ。OFFなら src を Dataset で追加して SRC を変更
    subprocess.run(['git','clone','--depth','1','-b','main',
                    'https://github.com/hidenori24/kaggle_ipynb', str(REPO)], check=True)
sys.path.insert(0, str(SRC))
from pseudo_labels import LABELS
from multilingual_labels import add_pseudo_labels_ml
import finetune as ft, torch
print('commit:', subprocess.run(['git','-C',str(SRC),'rev-parse','--short','HEAD'],capture_output=True,text=True).stdout.strip())

In [ ]:
train = pd.read_csv(COMP/'train.csv'); tseries = pd.read_csv(COMP/'train_series.csv')
test = pd.read_csv(COMP/'test.csv');   teseries = pd.read_csv(COMP/'test_series.csv')
print(train.shape, tseries.shape, test.shape)
train['has_true'] = train['ACL'].notna()   # 正解ラベルあり(疑似ラベルではない)
print('true-labeled:', int(train.has_true.sum()))
train = add_pseudo_labels_ml(train)        # 英語+トルコ語+ギリシャ語+ブルガリア語+スペイン語+ドイツ語+フランス語+オランダ語+クロアチア語のキーワード規則
print(train.lang.value_counts().to_dict())
# ---- ラベルの出どころ: 'llm' = 06_llm_labels が作ったソフトラベル(Dataset rsna-knee-labels)。無ければ 'rules'(キーワード規則)
LABEL_SOURCE = 'llm'
def find_dir(name):
    for base in [Path('/kaggle/input'), Path('/kaggle/input/datasets')]:
        if not base.exists(): continue
        for d in base.iterdir():
            if d.name == name: return d
            if d.is_dir() and d.name not in ('competitions', 'datasets'):
                for s in d.iterdir():
                    if s.name == name: return s
    return None
train['llm_ok'] = False
_d = find_dir('rsna-knee-labels') if LABEL_SOURCE == 'llm' else None
_files = sorted(_d.glob('llm_labels*.csv')) if _d is not None else []
if _files:                                                               # 複数のプロンプトで作ったラベルは平均する
    print('LLM label files:', [f.name for f in _files])
    stack = [pd.read_csv(f).drop_duplicates('StudyInstanceUID').set_index('StudyInstanceUID').reindex(train.StudyInstanceUID)[LABELS].values
             for f in _files]
    soft = np.nanmean(np.stack(stack), axis=0) if len(stack) > 1 else stack[0]
    ok = ~np.isnan(soft).any(1) & ~train.has_true.values                  # 正解ラベル付きは正解のまま
    for j, c in enumerate(LABELS):
        train.loc[ok, c] = soft[ok, j]
    train['llm_ok'] = ok
    print('LLM soft labels used for', int(ok.sum()), 'studies; mean per label:', dict(zip(LABELS, soft[ok].mean(0).round(2))))
else:
    LABEL_SOURCE = 'rules'
    print('LLM labels not found -> keyword rules')
# 言語ごとの疑似陽性率(本文は出さない)。英語と桁が大きく違う言語は規則が弱い疑い
print(train[~train.has_true].groupby('lang')[LABELS].mean().round(3).T)
# 診断: 規則の弱い言語(bg, tr)と未対応('other')の頻出語。本文は出さず、語の文書頻度だけを出す
import re
from collections import Counter
def top_words(mask, n):
    docs = train.loc[mask, 'Report'].fillna('').astype(str).str.lower()
    c = Counter(w for t in docs for w in set(re.findall(r'[^\W\d_]{4,}', t)))
    return c.most_common(n)
for lg, n in (('bg', 120), ('tr', 120), ('other', 150)):
    print(f'top words [{lg}] ({int((train.lang == lg).sum())} reports):', top_words(train.lang == lg, n))

# 診断: キーワード規則が、正解ラベル付き58件にどれだけ合っているか(規則ラベルは正解を見ずに作り直す)
from multilingual_labels import label_report_lang
tv = train[train.has_true]
R = np.array([[label_report_lang(r, l)[c] for c in LABELS] for r, l in zip(tv.Report.fillna(''), tv.lang)])
T = tv[LABELS].values.astype(int)
tp, fp, fn = (R & T).sum(0), (R & (1 - T)).sum(0), ((1 - R) & T).sum(0)
print('rule labels vs TRUE labels (n=%d, languages: %s)' % (len(tv), tv.lang.value_counts().to_dict()))
print(pd.DataFrame({'true_pos': T.sum(0), 'TP': tp, 'FP': fp, 'FN': fn}, index=LABELS))
print('micro precision %.3f  recall %.3f' % (tp.sum() / max(tp.sum() + fp.sum(), 1), tp.sum() / max(tp.sum() + fn.sum(), 1)))


In [ ]:
SIZE, K, EPOCHS, BS = 256, 24, 5, 6
RUN_TAG = 'f'      # 保存するモデルのファイル名 ft_model_<RUN_TAG>_s<seed>.pt。タグを変えれば、前の設定のモデルを消さずに、別設定のモデルを足せる(提出時は全モデルの予測を平均)
CENTER = 0.7       # 各seriesの中央70%のスライスからK枚を取る(端のスライスは膝が写っていない)
USE_META = True    # DICOMヘッダー(スキャナー・スライス厚・解像度など)もモデルに入力する
CROP = 0.85        # 視野(縦横)の中央85%を切り出してからSIZEにリサイズ(関節の細部に画素を回す)。1.0なら視野全体
ADJ = True         # 前後のスライスをRGBの3チャンネルに積む(事前学習の第1層をそのまま使い、スライス方向の文脈も見る)
NFOLD, ALPHA = 3, 0.5   # OOF予測の分割数と、ラベルへの混ぜ具合(0=元のラベルのまま)
ARCH = 'resnet18'  # 'resnet34' / 'resnet50' も可(その分遅い)
CACHE = Path('/kaggle/temp') if Path('/kaggle/temp').exists() else Path('/tmp')   # /kaggle/working に置くと出力に含まれてしまう
USE_LANGS = ['en', 'tr', 'el', 'bg', 'es', 'de', 'fr', 'nl', 'hr']   # 使う言語。英語のみで比較したいときは ['en']
sel = train.has_true | (train.llm_ok if LABEL_SOURCE == 'llm' else train.lang.isin(USE_LANGS))   # LLMラベルは言語を問わない
use = train[sel].reset_index(drop=True)   # 疑似ラベル + 正解ラベル付き
print('label source:', LABEL_SOURCE)
print(use.lang.value_counts().to_dict())
print('studies:', len(use), '| true-labeled:', int(use.has_true.sum()))
import shutil
dev = 'cuda' if torch.cuda.is_available() else 'cpu'; print('device', dev)
need = len(use) * 3 * K * SIZE * SIZE; free = shutil.disk_usage(CACHE).free
print(f'cache: {need / 1e9:.1f} GB needed, {free / 1e9:.1f} GB free'); assert free > 1.1 * need, 'not enough disk for the image cache: lower K or SIZE'
def probe(bs):   # 最大サイズの入力で1回だけ順伝播+逆伝播して、メモリが足りるか確かめる
    net = ft.KneeNet(len(LABELS), pretrained=False, meta_dim=ft.META_DIM if USE_META else 0, adj=ADJ, arch=ARCH).to(dev).train()
    x = torch.rand(bs, 3, K, SIZE, SIZE, device=dev); m = torch.ones(bs, 3, dtype=torch.bool, device=dev)
    md = torch.rand(bs, ft.META_DIM, device=dev) if USE_META else None
    with torch.autocast(device_type='cuda', enabled=dev == 'cuda'):
        loss = net(x, m, md).float().sum()
    loss.backward(); del net, x, loss
for _bs in [b for b in (BS, 6, 4, 3, 2, 1) if b <= BS]:   # 学習が始まってから(キャッシュに約75分かけた後)メモリ不足で落ちないように、先に確かめる
    try:
        if dev == 'cuda': probe(_bs)
        BS = _bs; break
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache(); print('out of memory at batch size', _bs)
print('batch size', BS)
X, M, META = ft.cache_volumes(use, tseries, COMP/'train_series', SIZE, K, CACHE, 'train', center=CENTER, crop=CROP)
META_ARG = META if USE_META else None
from dicom_meta import _MFR
print('manufacturers (siemens/ge/philips/...):', dict(zip(_MFR, META[:, 3 * ft.N_NUM:].sum(0).astype(int))),
      '| studies with header meta:', int((META[:, :ft.N_NUM] != 0).any(1).sum()))
print('planes available (Sag/Cor/Ax):', M.sum(0))
Y = use[LABELS].values.astype(np.float32)

In [ ]:
# 診断: 平面ごとのシリーズ数(1平面に複数のシリーズがあるか。あれば、複数シリーズを入れる余地がある)
g = tseries.groupby(['StudyInstanceUID', 'Anatomical_Plane']).size().unstack(fill_value=0)
print('series per plane (share of studies):'); print(g.apply(lambda s: s.value_counts(normalize=True)).fillna(0).round(3))
print(tseries.groupby(['Anatomical_Plane', 'Fluid_Sensitive', 'Fat_Suppression']).size())

In [ ]:
OUT = Path('model_out'); OUT.mkdir(exist_ok=True)
true_idx = np.where(use.has_true.values)[0]
pseudo_all = np.where(~use.has_true.values)[0]
rng = np.random.RandomState(123); perm = rng.permutation(pseudo_all)
folds = np.array_split(perm, NFOLD)

def train_one(tr, Ytr, seed, tag=''):
    r = np.random.RandomState(seed); tr = r.permutation(tr); n_va = int(0.08 * len(tr))
    va, tr = tr[:n_va], tr[n_va:]
    torch.manual_seed(seed); np.random.seed(seed)
    net, hist = ft.fit(X, M, Ytr, tr, {'pseudo': va, 'true': true_idx}, dev, epochs=EPOCHS, bs=BS, META=META_ARG, adj=ADJ, arch=ARCH)
    return net

# ---- 1. OOF予測(元ラベルで学習)
OOF = np.zeros_like(Y); fold_true = []
for f in range(NFOLD):
    held = folds[f]; tr = np.concatenate([folds[g] for g in range(NFOLD) if g != f])
    print(f'=== fold {f}: train {len(tr)} held-out {len(held)}')
    net = train_one(tr, Y, f)
    OOF[held] = ft.predict(net, X, M, held, dev, META=META_ARG)
    fold_true.append(ft.predict(net, X, M, true_idx, dev, META=META_ARG))
    del net; torch.cuda.empty_cache()
np.save(OUT/'oof.npy', OOF)
print(f'fold ensemble ({NFOLD} models, original labels): macro AUC vs TRUE {ft.macro_auc(Y[true_idx], np.mean(fold_true, axis=0)):.4f}')
print('OOF vs original labels (agreement, per label AUC of OOF against hard labels):')
print(ft.per_label_auc(Y[pseudo_all] >= 0.5, OOF[pseudo_all], LABELS).round(3))

In [ ]:
# ---- 2. ラベルの作り直し
Y2 = Y.copy()
Y2[pseudo_all] = (1 - ALPHA) * Y[pseudo_all] + ALPHA * OOF[pseudo_all]
print('mean per label  original:', Y[pseudo_all].mean(0).round(3)); print('mean per label  blended :', Y2[pseudo_all].mean(0).round(3))
# ---- 3. 新ラベルで本学習
N_SEEDS = 2
true_preds = []
for seed in range(N_SEEDS):
    print(f'=== final seed {seed}')
    net = train_one(pseudo_all, Y2, 100 + seed)
    torch.save({'state': net.state_dict(), 'size': SIZE, 'k': K, 'center': CENTER, 'crop': CROP, 'adj': ADJ, 'arch': ARCH, 'meta': USE_META, 'labels': LABELS}, OUT/f'ft_model_{RUN_TAG}_s{seed}.pt')
    true_preds.append(ft.predict(net, X, M, true_idx, dev, META=META_ARG)); del net; torch.cuda.empty_cache()
for i, P in enumerate(true_preds): print(f'seed {i}: macro AUC vs TRUE labels {ft.macro_auc(Y[true_idx], P):.4f}')
print(f'ensemble ({N_SEEDS} models, distilled labels): macro AUC vs TRUE labels {ft.macro_auc(Y[true_idx], np.mean(true_preds, axis=0)):.4f}')
print(ft.per_label_auc(Y[true_idx], np.mean(true_preds, axis=0), LABELS).round(3))